# nb09 — 5-Class Ablation (Base-Paper Protocol, Ensemble Pipeline)

Head-to-head with the base paper (Hoque & Seddiqui, 2025, *Frontiers in AI*) on **its own setup**:
the **44K Facebook source**, **5 classes**, **70/15/15** split. We apply **our main method** end-to-end:
single-task fine-tuning of the three encoders (BanglaBERT, MuRIL, XLM-R) with the project's final
recipe (uniform LR, class-balanced sampler, per-class loss tuning), then a **weighted-logit ensemble**
— exactly like NB05→NB06.

**No leakage:** fresh models are trained on the facebook-70% **train** split only; we never reuse the
main-pipeline checkpoints/logits (those saw facebook during their own training). Weights are optimised
on the facebook **val** (15%); the ensemble is reported on the facebook **test** (15%) for an apples-to-
apples comparison with the base paper's test-set numbers.

Base-paper targets: **5-class F1 = 89.23 %**; binary F1 = 93.61 % (compared via `non_bully` none-as-binary).


In [ ]:
# ── Install — assumed identical to NB05 (uncomment if needed) ──
# !pip install torch==2.4.1 transformers==4.44.2 scikit-learn==1.5.1 pandas==2.2.2 numpy==1.26.4 --quiet
print("deps assumed identical to NB05")

In [ ]:
import os, json, time, math, random, warnings
from collections import defaultdict
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import (f1_score, accuracy_score, matthews_corrcoef,
                             roc_auc_score, classification_report)
from scipy.optimize import minimize
warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"CUDA: {torch.cuda.is_available()} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CONFIG — 5-class ablation (mirrors NB05 FINAL single-task recipe)
# ═══════════════════════════════════════════════════════════════════
DEBUG          = True       # <-- run once True (smoke test), then set False
DEBUG_SAMPLES  = 256
DEBUG_EPOCHS   = 2

CONFIG = {
    "models": {"banglabert": "csebuetnlp/banglabert",
               "muril":      "google/muril-base-cased",
               "xlmr":       "xlm-roberta-base"},
    "max_length": 128, "batch_size": 16, "eval_batch_size": 32, "grad_accum_steps": 2, "num_workers": 0,
    "epochs": 8, "encoder_lr": 2e-5, "head_lr": 8e-5, "weight_decay": 0.01, "warmup_ratio": 0.10,
    "max_grad_norm": 1.0, "label_smoothing": 0.03,
    "focal_gamma_type5": 2.0, "class_weight_beta": 0.999,
    "use_class_balanced_sampler": True, "sampler_alpha": 0.5, "per_class_loss_multiplier": {},
    "dropout": 0.25, "head_hidden_dim": 384, "patience": 3, "use_fp16": torch.cuda.is_available(),
}
USE_LRDECAY = False

# PURE SINGLE-TASK: 5-class head only (base-paper Sub-task B). non_bully = none-as-binary.
TASK_CONFIG = {"type5": {"col": "label_type5", "num_classes": None, "loss_weight": 1.0, "monitor_weight": 1.0}}

CLEANED_CSV     = "../data/processed/benchmark_cleaned.csv"
FACEBOOK_SOURCE = "facebook_44001"
TEXT_COL        = "text_clean"
OUTPUT_DIR      = "../outputs/ablation_5class"
SPLIT_SEED      = 42
RUN_MODELS      = ["banglabert", "muril", "xlmr"]
RUN_SEEDS       = [42, 123, 456]      # 9 runs feed the ensemble
FORCE_RETRAIN   = False
os.makedirs(OUTPUT_DIR, exist_ok=True)
if DEBUG:
    CONFIG["epochs"] = DEBUG_EPOCHS; RUN_SEEDS = [42]
    print("⚠ DEBUG — subsampled, 1 seed, 2 epochs. Set DEBUG=False for the real run.")
print(f"OUTPUT_DIR: {OUTPUT_DIR} | runs: {len(RUN_MODELS)*len(RUN_SEEDS)} | sampler alpha={CONFIG['sampler_alpha']}")

In [ ]:
# ── Load facebook subset → 5 base-paper classes → 70/15/15 stratified split ──
df = pd.read_csv(CLEANED_CSV)
assert "source" in df.columns
fb = df[df["source"] == FACEBOOK_SOURCE].reset_index(drop=True).copy()
print(f"Facebook rows (post-dedup): {len(fb):,}  (base paper = 44,001)")
print("\nRaw label_type values:"); print(fb["label_type"].value_counts().to_string())

def map_5class(raw):
    if not isinstance(raw, str) or not raw.strip(): return "non_bully"
    s = raw.strip().lower().replace("_", " ")
    if "not bully" in s or s == "none" or "non" in s: return "non_bully"
    if "sexual" in s:                                  return "sexual"
    if "threat" in s or "calltoviolence" in s:         return "threat"
    if "religi" in s:                                  return "religious"
    if "troll" in s:                                   return "troll"
    return "__other__"

fb["label_type5"] = fb["label_type"].apply(map_5class)
dropped = int((fb["label_type5"] == "__other__").sum())
if dropped: print(f"\n⚠ Dropping {dropped:,} rows outside the 5 base-paper classes (dedup winners from other sources).")
fb = fb[fb["label_type5"] != "__other__"].reset_index(drop=True)
fb["label_binary"] = (fb["label_type5"] != "non_bully").astype(int)
print("\n5-class distribution:"); print(fb["label_type5"].value_counts().to_string())
assert fb["label_type5"].nunique() == 5, "Expected 5 classes — inspect raw values above and adjust map_5class()."

tr_idx, tmp = train_test_split(fb.index, test_size=0.30, random_state=SPLIT_SEED, stratify=fb["label_type5"])
va_idx, te_idx = train_test_split(tmp, test_size=0.50, random_state=SPLIT_SEED, stratify=fb.loc[tmp, "label_type5"])
train_df, val_df, test_df = (fb.loc[tr_idx].reset_index(drop=True),
                             fb.loc[va_idx].reset_index(drop=True),
                             fb.loc[te_idx].reset_index(drop=True))
if DEBUG:
    train_df = train_df.sample(min(DEBUG_SAMPLES, len(train_df)), random_state=SPLIT_SEED).reset_index(drop=True)
    val_df   = val_df.sample(min(DEBUG_SAMPLES//2, len(val_df)), random_state=SPLIT_SEED).reset_index(drop=True)
    test_df  = test_df.sample(min(DEBUG_SAMPLES//2, len(test_df)), random_state=SPLIT_SEED).reset_index(drop=True)
print(f"\nSplit  train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}  (70/15/15)")

def to_scalar(x): return x.item() if hasattr(x, "item") else x
label_encoders = {}
for tname, tcfg in TASK_CONFIG.items():
    col = tcfg["col"]
    vals = sorted(set(to_scalar(v) for v in pd.concat([train_df[col], val_df[col], test_df[col]]).dropna()), key=lambda z: str(z))
    label_encoders[tname] = {v: i for i, v in enumerate(vals)}; tcfg["num_classes"] = len(vals)
    print(f"  task '{tname}': {len(vals)} classes -> {list(label_encoders[tname].keys())}")
assert TASK_CONFIG["type5"]["num_classes"] == 5
with open(f"{OUTPUT_DIR}/label_encoders.json", "w", encoding="utf-8") as f:
    json.dump({k: {str(a): b for a, b in v.items()} for k, v in label_encoders.items()}, f, indent=2, ensure_ascii=False)

In [ ]:
# ── Dataset / model / loss / sampler — identical recipe to NB05 ─────────────
class CyberBullyDataset(Dataset):
    def __init__(self, df, tok, max_length, active_tasks, label_encoders, text_col):
        self.texts = df.reset_index(drop=True)[text_col].fillna("").astype(str).tolist()
        self.tok, self.maxlen = tok, max_length
        self.labels = {t: [int(label_encoders[t].get(to_scalar(v), -1)) for v in df[c["col"]].fillna("__m__")]
                       for t, c in active_tasks.items()}
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        enc = self.tok(self.texts[i], max_length=self.maxlen, truncation=True, padding=False)
        item = {k: torch.tensor(v, dtype=torch.long) for k, v in enc.items()}
        for t in self.labels: item[f"label_{t}"] = torch.tensor(self.labels[t][i], dtype=torch.long)
        return item

class MultiTaskCollator:
    def __init__(self, tok): self.tok = tok
    def __call__(self, fs):
        lk = [k for k in fs[0] if k.startswith("label_")]
        labels = {k: torch.stack([f[k] for f in fs]) for k in lk}
        tf = [{k: v for k, v in f.items() if not k.startswith("label_")} for f in fs]
        b = self.tok.pad(tf, padding=True, return_tensors="pt"); b.update(labels); return b

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.0):
        super().__init__(); self.gamma, self.weight, self.ls = gamma, weight, label_smoothing
    def forward(self, logits, t):
        ce = F.cross_entropy(logits, t, weight=self.weight, reduction="none", label_smoothing=self.ls)
        return (((1 - torch.exp(-ce)) ** self.gamma) * ce).mean()

class TaskHead(nn.Module):
    def __init__(self, h, n, dropout=0.25, inner=384):
        super().__init__(); inner = min(inner, h)
        self.net = nn.Sequential(nn.Dropout(dropout), nn.Linear(h, inner), nn.GELU(),
                                 nn.LayerNorm(inner), nn.Dropout(dropout), nn.Linear(inner, n))
    def forward(self, x): return self.net(x)

class MultiTaskTransformer(nn.Module):
    def __init__(self, model_name, active_tasks, dropout=0.25, head_hidden_dim=384):
        super().__init__(); self.encoder = AutoModel.from_pretrained(model_name)
        h = self.encoder.config.hidden_size
        self._tti = self.encoder.config.model_type.lower() not in ("roberta", "xlm-roberta")
        self.heads = nn.ModuleDict({t: TaskHead(h, c["num_classes"], dropout, head_hidden_dim) for t, c in active_tasks.items()})
    def _pool(self, out, mask):
        hs = out.last_hidden_state; cls = hs[:, 0]
        mean = (hs * mask.unsqueeze(-1).float()).sum(1) / mask.sum(1, keepdim=True).float().clamp(1)
        return 0.5 * cls + 0.5 * mean
    def forward(self, ids, mask, tti=None):
        kw = {"input_ids": ids, "attention_mask": mask}
        if self._tti and tti is not None: kw["token_type_ids"] = tti
        p = self._pool(self.encoder(**kw), mask); return {t: hd(p) for t, hd in self.heads.items()}

def get_uniform_params(model, enc_lr, head_lr, wd):
    nd = ["bias", "LayerNorm.weight", "LayerNorm.bias"]; g = []
    for grp, lr in [(model.encoder, enc_lr), (model.heads, head_lr)]:
        dec, ndec = [], []
        for n, p in grp.named_parameters():
            if p.requires_grad: (ndec if any(x in n for x in nd) else dec).append(p)
        g += [{"params": dec, "lr": lr, "weight_decay": wd}, {"params": ndec, "lr": lr, "weight_decay": 0.0}]
    return g

def build_class_weights(series, enc, beta=0.999, max_w=10.0, per_class_mult=None):
    mapped = series.map(enc).dropna().astype(int); n_cls = len(enc)
    counts = mapped.value_counts().sort_index(); w = np.ones(n_cls, dtype=np.float32)
    for i in range(n_cls):
        n = int(counts.get(i, 0))
        if n > 0: w[i] = 1.0 / max((1.0 - beta**n) / max(1.0 - beta, 1e-12), 1e-9)
    w = np.clip(w, w.min(), w.min() * max_w)
    if per_class_mult:
        for c, m in per_class_mult.items():
            if c in enc: w[enc[c]] *= float(m)
    w = w / w.mean(); return torch.tensor(w, dtype=torch.float32, device=device)

def make_balanced_sampler(df, col, enc, alpha=0.5):
    y = df[col].map(enc).fillna(-1).astype(int).values
    counts = np.bincount(y[y >= 0], minlength=len(enc)).astype(float); counts[counts == 0] = 1.0
    cw = (1.0 / counts) ** float(alpha)
    sw = np.where(y >= 0, cw[np.clip(y, 0, None)], 0.0)
    return WeightedRandomSampler(torch.as_tensor(sw, dtype=torch.double), num_samples=len(sw), replacement=True)
print("model + loss + sampler defined ✅")

In [ ]:
# ── Metrics + logits ────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate_tasks(model, loader, active_tasks):
    model.eval(); P={t:[] for t in active_tasks}; Y={t:[] for t in active_tasks}; PR={t:[] for t in active_tasks}
    for b in loader:
        b = {k: v.to(device) for k, v in b.items()}
        lg = model(b["input_ids"], b["attention_mask"], b.get("token_type_ids"))
        for t in active_tasks:
            y = b[f"label_{t}"].cpu().numpy(); pr = F.softmax(lg[t], -1).cpu().numpy(); m = y >= 0
            P[t].extend(pr[m].argmax(-1)); Y[t].extend(y[m]); PR[t].extend(pr[m])
    out = {}
    for t in active_tasks:
        y, p, pr = np.array(Y[t]), np.array(P[t]), np.array(PR[t])
        rec = {"macro_f1": round(float(f1_score(y, p, average="macro", zero_division=0)), 4),
               "weighted_f1": round(float(f1_score(y, p, average="weighted", zero_division=0)), 4),
               "accuracy": round(float(accuracy_score(y, p)), 4),
               "mcc": round(float(matthews_corrcoef(y, p)), 4)}
        try:
            rec["auroc"] = round(float(roc_auc_score(y, pr, multi_class="ovr", average="macro",
                                                     labels=list(range(pr.shape[1])))), 4)
        except Exception: rec["auroc"] = float("nan")
        out[t] = rec
    return out

@torch.no_grad()
def get_logits(model, loader, active_tasks):
    model.eval(); store = {t: [] for t in active_tasks}
    for b in loader:
        b = {k: v.to(device) for k, v in b.items()}
        lg = model(b["input_ids"], b["attention_mask"], b.get("token_type_ids"))
        for t in active_tasks: store[t].append(lg[t].cpu())
    return {t: torch.cat(v) for t, v in store.items()}
print("metrics defined ✅")

In [ ]:
# ── Train one (model, seed): single-task, sampler, monitor on type5 ─────────
def set_seed(s): random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def train_one(model_key, model_name, seed):
    set_seed(seed); sd = f"{OUTPUT_DIR}/{model_key}_seed{seed}"; os.makedirs(sd, exist_ok=True)
    tok = AutoTokenizer.from_pretrained(model_name); coll = MultiTaskCollator(tok)
    lkw = dict(collate_fn=coll, num_workers=CONFIG["num_workers"], pin_memory=torch.cuda.is_available())
    train_ds = CyberBullyDataset(train_df, tok, CONFIG["max_length"], TASK_CONFIG, label_encoders, TEXT_COL)
    if CONFIG["use_class_balanced_sampler"]:
        sampler = make_balanced_sampler(train_df, "label_type5", label_encoders["type5"], CONFIG["sampler_alpha"])
        train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], sampler=sampler, shuffle=False, drop_last=True, **lkw)
    else:
        train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, drop_last=True, **lkw)
    val_loader  = DataLoader(CyberBullyDataset(val_df,  tok, CONFIG["max_length"], TASK_CONFIG, label_encoders, TEXT_COL), batch_size=CONFIG["eval_batch_size"], shuffle=False, **lkw)
    test_loader = DataLoader(CyberBullyDataset(test_df, tok, CONFIG["max_length"], TASK_CONFIG, label_encoders, TEXT_COL), batch_size=CONFIG["eval_batch_size"], shuffle=False, **lkw)

    model = MultiTaskTransformer(model_name, TASK_CONFIG, CONFIG["dropout"], CONFIG["head_hidden_dim"]).to(device)
    crit = {t: FocalLoss(CONFIG.get(f"focal_gamma_{t}", 2.0),
                         build_class_weights(train_df[c["col"]], label_encoders[t], beta=CONFIG["class_weight_beta"],
                                             per_class_mult=CONFIG.get("per_class_loss_multiplier", {})),
                         CONFIG["label_smoothing"]) for t, c in TASK_CONFIG.items()}
    opt = torch.optim.AdamW(get_uniform_params(model, CONFIG["encoder_lr"], CONFIG["head_lr"], CONFIG["weight_decay"]))
    nstep = math.ceil(len(train_loader)/CONFIG["grad_accum_steps"]) * CONFIG["epochs"]
    sch = get_linear_schedule_with_warmup(opt, int(nstep*CONFIG["warmup_ratio"]), nstep)
    scaler = torch.amp.GradScaler("cuda") if (CONFIG["use_fp16"] and device.type == "cuda") else None

    best, noimp = -1.0, 0
    for ep in range(CONFIG["epochs"]):
        model.train(); opt.zero_grad(set_to_none=True); el, t0 = 0.0, time.time()
        for st, b in enumerate(train_loader, 1):
            b = {k: v.to(device) for k, v in b.items()}
            with torch.autocast(device_type=device.type, enabled=scaler is not None):
                lg = model(b["input_ids"], b["attention_mask"], b.get("token_type_ids"))
                loss = torch.tensor(0.0, device=device)
                for t, c in TASK_CONFIG.items():
                    y = b[f"label_{t}"]; m = y >= 0
                    if m.any():
                        tl = crit[t](lg[t][m], y[m])
                        if not (torch.isnan(tl) or torch.isinf(tl)): loss = loss + c["loss_weight"] * tl
                loss = loss / CONFIG["grad_accum_steps"]
            (scaler.scale(loss) if scaler else loss).backward()
            if st % CONFIG["grad_accum_steps"] == 0 or st == len(train_loader):
                if scaler: scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["max_grad_norm"])
                (scaler.step(opt), scaler.update()) if scaler else opt.step()
                sch.step(); opt.zero_grad(set_to_none=True)
            el += loss.item() * CONFIG["grad_accum_steps"]
        mon = evaluate_tasks(model, val_loader, TASK_CONFIG)["type5"]["macro_f1"]
        flag = ""
        if mon > best: best, noimp = mon, 0; torch.save(model.state_dict(), f"{sd}/best_model.pt"); flag = " ✅BEST"
        else: noimp += 1
        print(f"  Ep{ep+1:2}/{CONFIG['epochs']} loss={el/max(len(train_loader),1):.4f} val_type5_F1={mon:.4f} {(time.time()-t0)/60:.1f}min{flag}")
        if noimp >= CONFIG["patience"]: print(f"  early stop ep{ep+1}"); break

    model.load_state_dict(torch.load(f"{sd}/best_model.pt", map_location=device, weights_only=True))
    test_metrics = evaluate_tasks(model, test_loader, TASK_CONFIG)
    # Save val + test logits → the ensemble (NB06-style) consumes these. No leakage:
    # these come from a model trained ONLY on the facebook-70% train split.
    torch.save(get_logits(model, val_loader, TASK_CONFIG),  f"{sd}/val_logits.pt")
    torch.save(get_logits(model, test_loader, TASK_CONFIG), f"{sd}/test_logits.pt")
    res = {"model": model_key, "seed": seed, "best_val_type5_f1": round(best, 4), "test_metrics": test_metrics}
    json.dump(res, open(f"{sd}/results.json", "w"), indent=2)
    print(f"  -> single-model test type5 macroF1={test_metrics['type5']['macro_f1']:.4f}")
    return res
print("trainer defined ✅")

In [ ]:
# ── Run all (model × seed) ──────────────────────────────────────────────────
all_results = []
for mk in RUN_MODELS:
    for s in RUN_SEEDS:
        sd = f"{OUTPUT_DIR}/{mk}_seed{s}"
        if not FORCE_RETRAIN and os.path.exists(f"{sd}/results.json"):
            print(f"⏭  {mk} seed={s} done"); all_results.append(json.load(open(f"{sd}/results.json"))); continue
        print(f"\n▶ {mk} seed={s}")
        try: all_results.append(train_one(mk, CONFIG["models"][mk], s))
        except Exception as e:
            import traceback; print(f"❌ {mk} seed={s}: {e}"); traceback.print_exc()
        torch.cuda.empty_cache()
print(f"\n🎉 {len(all_results)}/{len(RUN_MODELS)*len(RUN_SEEDS)} runs complete")

In [ ]:
# ── ENSEMBLE (our main method): optimise weights on VAL, report on TEST ──────
import glob
runs = sorted(glob.glob(f"{OUTPUT_DIR}/*_seed*"))
val_logits, test_logits = {}, {}
for d in runs:
    n = os.path.basename(d)
    if os.path.exists(f"{d}/val_logits.pt") and os.path.exists(f"{d}/test_logits.pt"):
        val_logits[n]  = torch.load(f"{d}/val_logits.pt",  map_location="cpu", weights_only=False)
        test_logits[n] = torch.load(f"{d}/test_logits.pt", map_location="cpu", weights_only=False)
print(f"Loaded {len(val_logits)} runs for ensembling")

enc = label_encoders["type5"]
y_val  = np.array([enc.get(to_scalar(v), -1) for v in val_df["label_type5"]])
y_test = np.array([enc.get(to_scalar(v), -1) for v in test_df["label_type5"]])

def ens_logits(d, w):
    names = list(d.keys()); out = None
    for i, n in enumerate(names):
        l = d[n]["type5"].float(); out = w[i]*l if out is None else out + w[i]*l
    return out / sum(w)

def optimise(vd, y):
    names = list(vd.keys()); k = len(names)
    def neg(rw):
        w = np.abs(rw) + 1e-8
        return -f1_score(y, ens_logits(vd, w).argmax(-1).numpy(), average="macro", zero_division=0)
    best, bw = 1.0, None
    for _ in range(50):
        r = minimize(neg, np.random.dirichlet(np.ones(k)), method="Nelder-Mead",
                     options={"maxiter": 2000, "xatol": 1e-5})
        if r.fun < best: best, bw = r.fun, np.abs(r.x) + 1e-8
    bw = bw / bw.sum()
    print("Ensemble weights:"); [print(f"  {n}: {w:.4f}") for n, w in zip(names, bw)]
    print(f"VAL type5 macro-F1 (optimised): {-best:.4f}")
    return bw

ens_metrics = {}
if val_logits:
    W = optimise(val_logits, y_val)
    ens = ens_logits(test_logits, W); pr = torch.softmax(ens, -1).numpy(); yp = ens.argmax(-1).numpy()
    v = y_test >= 0; yt, yp, pr = y_test[v], yp[v], pr[v]; ncl = pr.shape[1]
    classes = sorted(enc, key=lambda k: enc[k]); none_id = enc.get("non_bully")
    pcf = f1_score(yt, yp, average=None, labels=list(range(ncl)), zero_division=0)
    try: au = roc_auc_score(yt, pr, multi_class="ovr", average="macro", labels=list(range(ncl)))
    except Exception: au = float("nan")
    ens_metrics = {
        "eval_split": "facebook_test_15pct",
        "type5_macro_f1": round(float(f1_score(yt, yp, average="macro", zero_division=0)), 4),
        "type5_weighted_f1": round(float(f1_score(yt, yp, average="weighted", zero_division=0)), 4),
        "type5_accuracy": round(float(accuracy_score(yt, yp)), 4),
        "type5_mcc": round(float(matthews_corrcoef(yt, yp)), 4),
        "type5_macro_auroc": round(float(au), 4),
        "non_bully_f1_as_binary": round(float(pcf[none_id]), 4) if none_id is not None else None,
    }
    print("\n" + "="*64); print("5-CLASS ENSEMBLE — facebook TEST (base-paper protocol)"); print("="*64)
    for k, val in ens_metrics.items(): print(f"  {k:24s}: {val}")
    print("\nPer-class report:"); print(classification_report(yt, yp, target_names=classes, zero_division=0))
    BASE = 0.8923
    print(f"\nOur 5-class ensemble Macro-F1 : {ens_metrics['type5_macro_f1']:.4f}  |  base paper : {BASE:.4f}  "
          f"|  Δ = {ens_metrics['type5_macro_f1']-BASE:+.4f}")
    json.dump(ens_metrics, open(f"{OUTPUT_DIR}/ensemble_5class_metrics.json", "w"), indent=2)
    print(f"✅ saved {OUTPUT_DIR}/ensemble_5class_metrics.json")
else:
    print("⚠ no logits — run the training cell first")

In [ ]:
# ── Per-model summary (mean±std across seeds), for the ablation table ────────
DISP = {"banglabert": "BanglaBERT", "muril": "MuRIL", "xlmr": "XLM-RoBERTa"}
rows = []
for mk in RUN_MODELS:
    rs = [r for r in all_results if r["model"] == mk]
    if not rs: continue
    row = {"Model": DISP[mk], "seeds": len(rs)}
    for key in ["macro_f1", "weighted_f1", "accuracy", "mcc", "auroc"]:
        vals = [r["test_metrics"]["type5"][key] for r in rs if not (isinstance(r["test_metrics"]["type5"].get(key), float) and np.isnan(r["test_metrics"]["type5"].get(key)))]
        row[f"type5_{key}"] = f"{np.mean(vals):.4f}±{np.std(vals):.4f}" if vals else "n/a"
    rows.append(row)
summary = pd.DataFrame(rows)
pd.set_option("display.width", 200, "display.max_columns", 30)
print(summary.to_string(index=False))
summary.to_csv(f"{OUTPUT_DIR}/per_model_5class_summary.csv", index=False)
print(f"\n✅ saved {OUTPUT_DIR}/per_model_5class_summary.csv")

---
**Reporting.** Headline = the **ensemble** 5-class Macro-F1 on the facebook 15% test (compared to the
base paper's 89.23 %). Per-model mean±std is the ablation table. The `non_bully` class F1 is the
none-as-binary figure (compare to 93.61 %). All models here were trained **only** on the facebook-70 %
train split — no main-pipeline checkpoint/logit is reused, so there is no leakage.
